<a href="https://colab.research.google.com/github/dhanya0412/major-project/blob/main/notebooks/01_week1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install -q datasets transformers accelerate bitsandbytes pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.3 MB/s eta 0:00:00


In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")

True
Tesla T4


In [3]:
!git clone https://github.com/dhanya0412/major-project.git

Cloning into 'major-project'...
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 7 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (7/7), 8.64 KiB | 4.32 MiB/s, done.


In [ ]:
%cd major-project/

/content/major-project


In [ ]:
!find . -maxdepth 2 -print

.
./README.md
./.git
./.git/hooks
./.git/branches
./.git/info
./.git/index
./.git/HEAD
./.git/description
./.git/packed-refs
./.git/objects
./.git/logs
./.git/refs
./.git/config


In [ ]:
!mkdir -p notebooks src results

In [ ]:
!find . -maxdepth 2 -print

.
./README.md
./results
./notebooks
./src
./.git
./.git/hooks
./.git/branches
./.git/info
./.git/index
./.git/HEAD
./.git/description
./.git/packed-refs
./.git/objects
./.git/logs
./.git/refs
./.git/config


In [1]:
!pwd


/content


In [2]:
!ls

sample_data


In [4]:
!ls -lah

total 20K
drwxr-xr-x 1 root root 4.0K Sep 18 10:33 .
drwxr-xr-x 1 root root 4.0K Sep 18 10:27 ..
drwxr-xr-x 4 root root 4.0K Sep  4 13:32 .config
drwxr-xr-x 4 root root 4.0K Sep 18 10:33 major-project
drwxr-xr-x 1 root root 4.0K Sep  4 13:32 sample_data


In [5]:
%cd /content/major-project

/content/major-project


In [7]:
import torch
import transformers
import datasets
import accelerate
import bitsandbytes

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

PyTorch: 2.11.0+cu128
Transformers: 5.16.1
Datasets: 4.8.5
CUDA: True
GPU: Tesla T4


In [8]:
from datasets import load_dataset

ds = load_dataset("lmms-lab/HallusionBench")

print(ds)

README.md:   0%|          | 0.00/2.66k [00:00<?, ?B/s]

data/image-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  147MB            

data/image-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/non_image-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 22.9kB            

data/non_image-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating image split:   0%|          | 0/951 [00:00<?, ? examples/s]

Generating non_image split:   0%|          | 0/178 [00:00<?, ? examples/s]

DatasetDict({
    image: Dataset({
        features: ['category', 'subcategory', 'visual_input', 'set_id', 'figure_id', 'sample_note', 'question_id', 'question', 'gt_answer_details', 'gt_answer', 'filename', 'image'],
        num_rows: 951
    })
    non_image: Dataset({
        features: ['category', 'subcategory', 'visual_input', 'set_id', 'figure_id', 'sample_note', 'question_id', 'question', 'gt_answer_details', 'gt_answer', 'filename', 'image'],
        num_rows: 178
    })
})


In [9]:
example = ds["image"][0]
print(example)

{'category': 'VS', 'subcategory': 'chart', 'visual_input': '1', 'set_id': '0', 'figure_id': '1', 'sample_note': 'import', 'question_id': '0', 'question': 'Is China, Hongkong SAR, the leading importing country of gold, silverware, and jewelry with the highest import value in 2018?', 'gt_answer_details': 'Switzerland is the leading importing country of gold, silverware, and jewelry with the highest import value in 2018?', 'gt_answer': '0', 'filename': './VS/chart/0_1.png', 'image': <PIL.PngImagePlugin.PngImageFile image mode=P size=986x762 at 0x78EB6AE59550>}


In [10]:
print(ds.keys())

dict_keys(['image', 'non_image'])


In [11]:
for split in ds:
    print("\n====================")
    print("SPLIT:", split)
    print("ROWS:", len(ds[split]))
    print("COLUMNS:", ds[split].column_names)


SPLIT: image
ROWS: 951
COLUMNS: ['category', 'subcategory', 'visual_input', 'set_id', 'figure_id', 'sample_note', 'question_id', 'question', 'gt_answer_details', 'gt_answer', 'filename', 'image']

SPLIT: non_image
ROWS: 178
COLUMNS: ['category', 'subcategory', 'visual_input', 'set_id', 'figure_id', 'sample_note', 'question_id', 'question', 'gt_answer_details', 'gt_answer', 'filename', 'image']


In [12]:
example = ds["image"][0]
print({k: v for k, v in example.items() if k != "image"})
print(type(example["image"]), example["image"].size)

{'category': 'VS', 'subcategory': 'chart', 'visual_input': '1', 'set_id': '0', 'figure_id': '1', 'sample_note': 'import', 'question_id': '0', 'question': 'Is China, Hongkong SAR, the leading importing country of gold, silverware, and jewelry with the highest import value in 2018?', 'gt_answer_details': 'Switzerland is the leading importing country of gold, silverware, and jewelry with the highest import value in 2018?', 'gt_answer': '0', 'filename': './VS/chart/0_1.png'}
<class 'PIL.PngImagePlugin.PngImageFile'> (986, 762)


In [13]:
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
import torch

model_id = "llava-hf/llava-1.5-7b-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(model_id)
model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/70.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

In [14]:
example = ds["image"][0]
image = example["image"]
question = example["question"]

prompt = f"USER: <image>\n{question} Please answer with only Yes or No. ASSISTANT:"

inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=20)
response = processor.decode(output[0], skip_special_tokens=True)

print("Question:", question)
print("Ground truth:", example["gt_answer"])
print("Model response:", response)

Question: Is China, Hongkong SAR, the leading importing country of gold, silverware, and jewelry with the highest import value in 2018?
Ground truth: 0
Model response: USER:  
Is China, Hongkong SAR, the leading importing country of gold, silverware, and jewelry with the highest import value in 2018? Please answer with only Yes or No. ASSISTANT: Yes
